In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 5GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

from matplotlib import pyplot as plt
import seaborn as sns
import gzip

test_file = '../input/avazu-ctr-prediction/test.gz'
samplesubmision_file = '../input/avazu-ctr-prediction/sampleSubmission.gz'

?????Row?????? (??4???) ,Pandas DataFrame???????????,??????100??????????,???????????,????5%???,??????train set?

In [ ]:
chunksize = 10 ** 6
num_of_chunk = 0
train = pd.DataFrame()
    
for chunk in pd.read_csv('../input/avazu-ctr-train/train.csv', chunksize=chunksize):
    num_of_chunk += 1
    train = pd.concat([train, chunk.sample(frac=.05, replace=False, random_state=123)], axis=0)
    print('Processing Chunk No. ' + str(num_of_chunk))     
    
train.reset_index(inplace=True)

# ??train ????,????df????????
train_len = len(train)
train_len

??test????????,??train, test ?????df,????????????

In [ ]:
df = pd.concat([train, pd.read_csv(test_file, compression='gzip')]).drop(['index', 'id'], axis=1)

??????????10??????????,?????11????,?????????,?weekday????????????hour??????,???weekday?

?,?????????????????,????????????????????24,?????????12???,?????????

In [ ]:
# ?????hour??????????function
def get_date(hour):
    y = '20'+str(hour)[:2]
    m = str(hour)[2:4]
    d = str(hour)[4:6]
    return y+'-'+m+'-'+d

# ??weekday??,?hour?????
df['weekday'] = pd.to_datetime(df.hour.apply(get_date)).dt.dayofweek.astype(str)

# ?????hour????????function
def tran_hour(x):
    x = x % 100
    while x in [23,0]:
        return '23-01'
    while x in [1,2]:
        return '01-03'
    while x in [3,4]:
        return '03-05'
    while x in [5,6]:
        return '05-07'
    while x in [7,8]:
        return '07-09'
    while x in [9,10]:
        return '09-11'
    while x in [11,12]:
        return '11-13'
    while x in [13,14]:
        return '13-15'
    while x in [15,16]:
        return '15-17'
    while x in [17,18]:
        return '17-19'
    while x in [19,20]:
        return '19-21'
    while x in [21,22]:
        return '21-23'

# ?hour?????
df['hour'] = df.hour.apply(tran_hour)

In [ ]:
# ??????
df.info()

???????value count,??int?????,??value????4,333?value count,???600?????????,??????????????,?????????,??Object??????

In [ ]:
len_of_feature_count = []
for i in df.columns[2:23].tolist():
    print(i, ':', len(df[i].astype(str).value_counts()))
    len_of_feature_count.append(len(df[i].astype(str).value_counts()))

In [ ]:
# ????list,???????????????list
need_tran_feature = df.columns[2:4].tolist() + df.columns[13:23].tolist()

# ????????object??
for i in need_tran_feature:
    df[i] = df[i].astype(str)

?????value count??,??????????,?????? one-hot??,??????????????????value count?10??10,????10??,?????????

??????,????????????,???????very_high, higher, mid, lower, very_low,?5????

In [ ]:
obj_features = []

for i in range(len(len_of_feature_count)):
    if len_of_feature_count[i] > 10:
        obj_features.append(df.columns[2:23].tolist()[i])
obj_features

In [ ]:
df_describe = df.describe()
df_describe

In [ ]:
def obj_clean(X):
    # ??????????function,??????????

    def get_click_rate(x):
        # ??????????function
        temp = train[train[X.columns[0]] == x]
        res = round((temp.click.sum() / temp.click.count()),3)
        return res

    def get_type(V, str):
        # ????????????????function
        very_high = df_describe.loc['mean','click'] + 0.04
        higher = df_describe.loc['mean','click'] + 0.02
        lower = df_describe.loc['mean','click'] - 0.02
        very_low = df_describe.loc['mean','click'] - 0.04

        vh_type = V[V[str] > very_high].index.tolist()
        hr_type = V[(V[str] > higher) & (V[str] < very_high)].index.tolist()
        vl_type = V[V[str] < very_low].index.tolist()
        lr_type = V[(V[str] < lower) & (V[str] > very_low)].index.tolist()

        return vh_type, hr_type, vl_type, lr_type

    def clean_function(x):
        # ??????????????function
        # ??????:?????????  4% ?very_high(low), ????????? 2%?higher (lower)
        while x in type_[0]:
            return 'very_high'
        while x in type_[1]:
            return 'higher'
        while x in type_[2]:
            return 'very_low'
        while x in type_[3]:
            return 'lower'
        return 'mid'
        
    print('Run: ', X.columns[0])
    fq = X[X.columns[0]].value_counts()
    # ??????????????
    # ???,??????????????,???????;???????????,??????????1000 row???????
    if len(fq) > 1000:
        fq = fq[:1000]

    # ????????dataframe,??index?????????
    fq = pd.DataFrame(fq)
    fq['new_column'] = fq.index    

    # ??index??get_click_rate function,???????????
    fq['click_rate'] = fq.new_column.apply(get_click_rate)

    # ?? get_type function??????,??????list,????????clean_function??
    type_ = get_type(fq, 'click_rate')

    # ?? clean_funtion funtion,??????????
    return X[X.columns[0]].apply(clean_function)

# ??for ???????????? obj_clean function
for i in obj_features:    
    df[[i]] = obj_clean(df[[i]])

df

In [ ]:
# ????????????
for i in df.columns:
    sns.countplot(x = i, hue = "click", data = df)
    plt.show()

????????????,?? ['device_id', 'C14', 'C17', 'C19', 'C20', 'C21'] ????,????value,????????????,?????????????

In [ ]:
df.drop(['device_id', 'C14', 'C17', 'C19', 'C20', 'C21'], axis=1, inplace=True)

In [ ]:
# ??????? one-hot ??
df = pd.get_dummies(df)

# ??????df???,???train, test????
train = df[:train_len]
test = df[train_len:]

In [ ]:
# # ?????train, test ?????,??????????????

# train.to_csv('new_train.csv', index=False)
# test.to_csv('new_test.csv', index=False)

In [ ]:
# # ??????train, test ???,??????????????
# train = pd.read_csv('new_train.csv')
# test = pd.read_csv('new_test.csv')

In [ ]:
del df

??????????,????label?????????17%??,??????,???????????????????????xgboost???,????????,????GPU?????????

???????,????????,?????100????,?Grid Search???????????,????xgboost????????????????????,????label??????,?????label??????????50%????,???????????????label????,????ROC_AUC ????????

In [ ]:
# ?train????,???0????,????????1??????,??????????????50%????
pre_X = train[train['click'] == 0].sample(n=len(train[train['click'] == 1]), random_state=111)
pre_X = pd.concat([pre_X, train[train['click'] == 1]]).sample(frac=1)
pre_y = pre_X[['click']]
pre_X.drop(['click'], axis=1, inplace=True)
test.drop(['click'], axis=1, inplace=True)

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

# ????????????????
pre_X_train, pre_X_test, pre_y_train, pre_y_test = train_test_split(pre_X, pre_y, test_size=0.20, stratify=pre_y, random_state=1)

In [ ]:
# ??Grid Search??,??100?????????
params = {"criterion":["gini", "entropy"], "max_depth":range(1,20)}
grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid=params, scoring='roc_auc', cv=100, verbose=1, n_jobs=-1)
grid_search.fit(pre_X_train, pre_y_train)
grid_search.best_score_, grid_search.best_estimator_, grid_search.best_params_

In [ ]:
# ??Grid Search????????????,?fit???? (????)
tree = grid_search.best_estimator_
tree.fit(pre_X,pre_y)

# ??????,??????????
feature_importances = pd.DataFrame(tree.feature_importances_)
feature_importances.index = pre_X_train.columns
feature_importances = feature_importances.sort_values(0,ascending=False)
feature_importances

In [ ]:
# ?????????????,??????????????????1/3
pre_X_train = pre_X_train[feature_importances.index[:int(len(feature_importances)/3)]]
pre_X_test = pre_X_test[feature_importances.index[:int(len(feature_importances)/3)]]

In [ ]:
# ??33%?????????Grid Search??
params = {"criterion":["gini", "entropy"], "max_depth":range(1,12)}
grid_search = GridSearchCV(DecisionTreeClassifier(), param_grid=params, scoring='roc_auc', cv=100, verbose=1, n_jobs=-1)
grid_search.fit(pre_X_train, pre_y_train)
grid_search.best_score_, grid_search.best_estimator_, grid_search.best_params_

In [ ]:
# ???????????,??????????????????1/3
pre_X = pre_X[feature_importances.index[:int(len(feature_importances)/3)]]

# ??Grid Search????????????,?fit???? (????)
tree = grid_search.best_estimator_
tree.fit(pre_X,pre_y)

# ??????,??????????
feature_importances = pd.DataFrame(tree.feature_importances_)
feature_importances.index = pre_X_train.columns
feature_importances = feature_importances.sort_values(0,ascending=False)
feature_importances

In [ ]:
# ?????????,?????? .005?????
feature_len = len(feature_importances[feature_importances[feature_importances.columns[0]] > 0.005])

# ??????Train Set ? Test set???
y = train[['click']]
X = train[feature_importances[:feature_len].index]
test = test[feature_importances[:feature_len].index]

In [ ]:
from xgboost import XGBClassifier

# ??xgboost ??,??????????????????xgboost ??,????????????????
model = XGBClassifier(tree_method = 'gpu_hist', n_jobs=-1, n_estimators=500, max_depth=11)
model.fit(X,y.values.ravel())
y_pred = model.predict(X)
print("Roc_auc_score: ",roc_auc_score(y,y_pred)*100,"%")

# ??????,??????
confmat = confusion_matrix(y_true=y, y_pred=y_pred, labels=[0, 1])

fig, ax = plt.subplots(figsize=(2.5, 2.5))
ax.matshow(confmat, cmap=plt.cm.Blues, alpha=0.3)
for i in range(confmat.shape[0]):
    for j in range(confmat.shape[1]):
        ax.text(x=j, y=i, s=confmat[i, j], va='center', ha='center')

plt.xlabel('Predicted label')
plt.ylabel('True label')

plt.tight_layout()
plt.show()

# ??submission?????
submission = pd.read_csv(samplesubmision_file, compression='gzip', index_col='id')
submission[submission.columns[0]] = model.predict_proba(test)[:,1]
submission.to_csv('submission.csv')

??????? .40560;Leader Board Top 1 ??? 0.37913